# Deep Learning 047 — Backpropagation in a CNN, Part 1

A deliberately tiny network — 6×6 image → conv → ReLU → max pool → flatten → one
output neuron — with **15 trainable parameters**, few enough to check every one.
The dense half needs **nothing new**: `dL/dZ2 = A2 − y`, exactly as in lesson 016.

| Part | What we check |
|---|---|
| A | the shapes: 6×6 → 4×4 → 4×4 → 2×2 → 4×1 → scalar |
| B | all fifteen parameters |
| C | the sigmoid/cross-entropy cancellation, numerically |
| D | the dense-side gradients against finite differences |

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 6x6 image -> conv 3x3 -> ReLU -> maxpool 2x2 -> flatten(4) -> dense -> sigmoid -> BCE
X = rng.normal(size=(6, 6))
y = 1.0
W1 = rng.normal(size=(3, 3)) * 0.5     # 9 parameters
b1 = 0.1                               # 1
W2 = rng.normal(size=4) * 0.5          # 4
b2 = -0.2                              # 1

def forward(X, W1, b1, W2, b2):
    Z1 = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            Z1[i, j] = np.sum(X[i:i+3, j:j+3] * W1) + b1
    A1 = np.maximum(0, Z1)                       # ReLU
    P = np.zeros((2, 2)); idx = {}
    for i in range(2):
        for j in range(2):
            win = A1[i*2:i*2+2, j*2:j*2+2]
            P[i, j] = win.max()
            a, b = np.unravel_index(win.argmax(), win.shape)
            idx[(i, j)] = (i*2 + a, j*2 + b)     # remember which element won
    flat = P.reshape(-1)
    Z2 = float(np.dot(flat, W2) + b2)
    A2 = 1 / (1 + np.exp(-Z2))
    loss = -(y * np.log(A2) + (1 - y) * np.log(1 - A2))
    return loss, dict(Z1=Z1, A1=A1, P=P, idx=idx, flat=flat, Z2=Z2, A2=A2)

## Part A — Shapes

In [ ]:
loss, c = forward(X, W1, b1, W2, b2)
print(f"{'input':<10}{X.shape}")
print(f"{'conv Z1':<10}{c['Z1'].shape}")
print(f"{'ReLU A1':<10}{c['A1'].shape}")
print(f"{'pool P':<10}{c['P'].shape}")
print(f"{'flatten':<10}{c['flat'].shape}")
print(f"{'Z2':<10}scalar -> A2 {c['A2']:.4f}, loss {loss:.4f}")
assert c['Z1'].shape == (4, 4) and c['P'].shape == (2, 2) and c['flat'].shape == (4,)

## Part B — Fifteen parameters

In [ ]:
counts = {"W1 (conv filter)": W1.size, "b1": 1, "W2 (dense)": W2.size, "b2": 1}
for k, v in counts.items():
    print(f"{k:<20}{v:>3}")
print(f"{'total':<20}{sum(counts.values()):>3}")
assert sum(counts.values()) == 15
print("\nReLU, pooling and flatten have no parameters - but they still route gradient.")

## Part C — The cancellation

With a sigmoid output and binary cross-entropy, the two derivatives cancel and
leave `A2 − y`. Check it numerically rather than taking it on trust.

In [ ]:
A2, Z2 = c["A2"], c["Z2"]

# dL/dA2 for BCE, times dA2/dZ2 for sigmoid
dL_dA2 = -(y / A2) + (1 - y) / (1 - A2)
dA2_dZ2 = A2 * (1 - A2)
chained = dL_dA2 * dA2_dZ2

print("dL/dA2 x dA2/dZ2 =", round(float(chained), 10))
print("A2 - y           =", round(float(A2 - y), 10))
assert np.isclose(chained, A2 - y)
print("\nSame as lesson 016, symbol for symbol. The CNN changed nothing here.")

## Part D — The dense-side gradients

In [ ]:
def numerical(param_setter, value, eps=1e-6):
    """Finite-difference gradient of the loss with respect to one scalar."""
    up = param_setter(value + eps); down = param_setter(value - eps)
    return (up - down) / (2 * eps)

dZ2 = c["A2"] - y
dW2_analytic = dZ2 * c["flat"]
db2_analytic = dZ2

dW2_numeric = []
for i in range(4):
    def setter(v, i=i):
        W2c = W2.copy(); W2c[i] = v
        return forward(X, W1, b1, W2c, b2)[0]
    dW2_numeric.append(numerical(setter, W2[i]))
db2_numeric = numerical(lambda v: forward(X, W1, b1, W2, v)[0], b2)

print(f"{'':>6}{'analytic':>14}{'numerical':>14}")
for i in range(4):
    print(f"W2[{i}]{dW2_analytic[i]:>14.8f}{dW2_numeric[i]:>14.8f}")
print(f"b2   {db2_analytic:>14.8f}{db2_numeric:>14.8f}")
assert np.allclose(dW2_analytic, dW2_numeric, atol=1e-6)
assert np.isclose(db2_analytic, db2_numeric, atol=1e-6)
print("\nThe dense half of a CNN is an ANN. Part 2 does the three new operations.")

## What to take away

- **6×6 → 4×4 → 4×4 → 2×2 → 4×1 → scalar.** Tracking shapes is most of the work.
- **Fifteen parameters**: W1 (9), b1 (1), W2 (4), b2 (1).
- **`dL/dZ2 = A2 − y`** — the sigmoid/cross-entropy cancellation, unchanged.
- ReLU, pooling and flatten have no parameters but must still pass gradient back.

## Exercises

1. Add a second output unit. How many parameters now, and what shape is dW2?
2. Change the pooling window to 3×3 (stride 1). What shape does the flatten give?
3. Why can the dense side be written without knowing anything about convolution?